In [ ]:
pra_raw = session.table(f"{DB}.{RAW}.STREAM_T_PERSON_ROLE_ASSIGNMENTS").filter(
    col("METADATA$ACTION") == "INSERT"
).drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID").cache_result()
print("Stream read complete")

In [ ]:
# Single .select() with all transformations
pra_clean = pra_raw.select(
    col("PRA1_ID"),
    col("POI_POI_ID"),
    upper(trim(col("ROLE_TYPE"))).alias("ROLE_TYPE"),
    col("ADD_TS"),
    coalesce(when(trim(col("ADD_USER")) == lit(""), lit("SYSTEM")).otherwise(trim(col("ADD_USER"))), lit("SYSTEM")).alias("ADD_USER"),
    col("START_DATE"),
    col("END_DATE"),
    col("TICKLER_ID"),
    coalesce(when(trim(col("MOD_USER")) == lit(""), lit("SYSTEM")).otherwise(trim(col("MOD_USER"))), lit("SYSTEM")).alias("MOD_USER"),
    col("MOD_TS"),
    coalesce(when(trim(col("ROLE_SUB_TYPE")) == lit(""), lit("NA")).otherwise(trim(col("ROLE_SUB_TYPE"))), lit("NA")).alias("ROLE_SUB_TYPE"),
    col("LOADED_DATE"),
    col("DELETED_DATE"),
    col("LOAD_TIMESTAMP").alias("RAW_LOAD_TIMESTAMP"),
    col("SOURCE_FILE_NAME")
)
print("Transformations applied")

In [ ]:
valid_pra = pra_clean.filter(col("PRA1_ID").is_not_null())
invalid_pra = pra_clean.filter(col("PRA1_ID").is_null()).with_column("ERROR_REASON", lit("PRA1_ID_NULL"))
print("Valid/invalid split defined")

In [ ]:
pra_final = valid_pra.with_column(
    "CHECKSUM",
    sha2(concat_ws(lit("|"),
        coalesce(col("PRA1_ID").cast("string"), lit("")),
        coalesce(col("POI_POI_ID").cast("string"), lit("")),
        coalesce(col("ROLE_TYPE"), lit("")),
        coalesce(col("ROLE_SUB_TYPE"), lit("")),
        coalesce(col("ADD_USER"), lit("")),
        coalesce(col("MOD_USER"), lit("")),
        coalesce(col("START_DATE").cast("string"), lit("")),
        coalesce(col("END_DATE").cast("string"), lit("")),
        coalesce(col("TICKLER_ID").cast("string"), lit(""))
    ), 256)
).with_column("STAGING_LOADED_TIMESTAMP", current_timestamp())

pra_final.write.save_as_table(f"{DB}.{STG}.{STG_PERSON_ROLE_ASSIGNMENTS}", mode="truncate")
print(f"PRA loaded into {STG}.{STG_PERSON_ROLE_ASSIGNMENTS}")

In [ ]:
invalid_count = invalid_pra.count()
if invalid_count > 0:
    invalid_pra.select(
        lit("T_PERSON_ROLE_ASSIGNMENTS").alias("TABLE_NAME"), col("ERROR_REASON"),
        col("SOURCE_FILE_NAME"), col("RAW_LOAD_TIMESTAMP")
    ).write.save_as_table(f"{DB}.{STG}.{INVALID_RECORDS}", mode="append")
    print(f"Invalid records saved: {invalid_count}")
else:
    print("No invalid records")

In [ ]:
rows_processed = session.table(f"{DB}.{STG}.{STG_PERSON_ROLE_ASSIGNMENTS}").count()
status = 'SUCCESS' if invalid_count == 0 else 'PARTIAL_SUCCESS'

session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    str(uuid.uuid4()), 'STG_PERSON_ROLE_ASSIGNMENTS_LOAD', 'STAGING',
    datetime.now(), datetime.now(), rows_processed, invalid_count, status, None)
session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status, 'STG_PERSON_ROLE_ASSIGNMENTS_LOAD', 'STAGING',
    f'PRA staging completed. Rows: {rows_processed}, Failed: {invalid_count}')
print(f"STG_PRA complete | Valid: {rows_processed} | Invalid: {invalid_count}")